# 🌐 Multilingual Hate Speech Detection — Robust BERT Fine-Tuning

> **Model:** `bert-base-multilingual-cased` (mBERT)  
> **Task:** Binary — `0 = not hate` · `1 = hate`  
> **Languages:** English · French · Arabic (Tunisian / Egyptian dialects + Classical)  
> **Hardware:** Kaggle T4 × 2 (DataParallel)  

### ⚠️ Anti-Overfitting Measures Applied
| Problem | Fix |
|---|---|
| Suspicious lexical features (profanity flags, keyword lists) | **Removed** — only raw text is used |
| Model memorising training set | Dropout 0.3 on classifier head + weight decay 0.05 |
| Learning too fast | Lower LR (1e-5) + warmup 15% |
| No early stopping | Val F1 patience=2 — saves best checkpoint and halts |
| Data leakage | Split **before** any feature engineering; stratified by label |
| Class-trivial patterns | Minimal text cleaning — no hate-keyword stripping |

---
```
0. Setup → 1. EDA → 2. Leak Audit → 3. Preprocessing → 4. Fine-Tuning → 5. Evaluation
```

## ⚙️ 0. Environment Setup

In [ ]:
!pip install -q transformers==4.40.0 accelerate==0.29.3 scikit-learn seaborn wordcloud

import os, gc, re, json, warnings, random
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from wordcloud import WordCloud

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.cuda.amp import autocast, GradScaler

from transformers import (
    BertTokenizerFast,
    BertForSequenceClassification,
    BertConfig,
    get_linear_schedule_with_warmup,
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve,
    f1_score, accuracy_score,
    precision_recall_curve, average_precision_score,
    matthews_corrcoef,
)

warnings.filterwarnings('ignore')

# ── Reproducibility ──────────────────────────────────────────────
SEED = 42
def set_seed(s=SEED):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False
set_seed()

# ── Device ───────────────────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
N_GPUS = torch.cuda.device_count()
print(f'✅ Device : {DEVICE}  |  GPUs : {N_GPUS}')
for i in range(N_GPUS):
    p = torch.cuda.get_device_properties(i)
    print(f'   GPU {i} : {p.name}  |  {p.total_memory/1e9:.1f} GB')

# ── Plot theme ───────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': '#0d1117', 'axes.facecolor':  '#0d1117',
    'axes.edgecolor':   '#30363d', 'axes.labelcolor': '#c9d1d9',
    'xtick.color':      '#c9d1d9', 'ytick.color':     '#c9d1d9',
    'text.color':       '#f0f6fc', 'grid.color':      '#21262d',
    'grid.linestyle':   '--',      'font.family':     'monospace',
})
C = {'blue':'#58a6ff','green':'#3fb950','red':'#f78166',
     'purple':'#d2a8ff','orange':'#ffa657','teal':'#39d353'}
PALETTE   = list(C.values())
LANG_CLR  = {'en': C['blue'], 'fr': C['purple'], 'ar': C['orange']}
LABEL_CLR = {0: C['green'], 1: C['red']}
print('✅ Setup complete')

---
## 📊 1. Load & EDA

In [ ]:
# ── 1.1  Load — set your path here ───────────────────────────────
DATA_PATH = ''   # ← paste your Kaggle dataset path here

df = pd.read_csv(DATA_PATH)
df.columns = df.columns.str.strip().str.lower()

# ── Identify columns ─────────────────────────────────────────────
# Expected: 'text', 'label' (0/1), 'lang' (en/fr/ar)
# Adjust these names if your CSV uses different ones
TEXT_COL  = 'text'
LABEL_COL = 'label'
LANG_COL  = 'lang'

df[TEXT_COL]  = df[TEXT_COL].astype(str)
df[LABEL_COL] = df[LABEL_COL].astype(int)
df[LANG_COL]  = df[LANG_COL].astype(str).str.strip().str.lower()

print('=' * 55)
print(' DATASET OVERVIEW')
print('=' * 55)
print(f'  Rows           : {len(df):,}')
print(f'  Columns        : {list(df.columns)}')
print(f'  Missing values : {df.isnull().sum().sum()}')
print(f'  Duplicates     : {df.duplicated(subset=[TEXT_COL]).sum():,}')
print()
print('Label distribution:')
vc = df[LABEL_COL].value_counts().sort_index()
for lbl, cnt in vc.items():
    print(f'  {lbl} : {cnt:,}  ({cnt/len(df)*100:.1f}%)')
print()
print('Language distribution:')
print(df[LANG_COL].value_counts())
print()
print('Language × Label cross-tab:')
print(pd.crosstab(df[LANG_COL], df[LABEL_COL], margins=True))

In [ ]:
# ── 1.2  Drop exact duplicates (same text + same label) ──────────
before = len(df)
df = df.drop_duplicates(subset=[TEXT_COL, LABEL_COL]).reset_index(drop=True)
print(f'Dropped {before - len(df):,} exact duplicates  →  {len(df):,} rows remain')

# Detect conflicting labels (same text, different labels) — data quality issue
conflicts = df.groupby(TEXT_COL)[LABEL_COL].nunique()
n_conflicts = int((conflicts > 1).sum())
print(f'Conflicting labels (same text, different label): {n_conflicts}')
if n_conflicts > 0:
    # Keep majority label for each text
    df = (
        df.groupby(TEXT_COL)
          .apply(lambda g: g.loc[g[LABEL_COL] == g[LABEL_COL].mode()[0]])
          .reset_index(drop=True)
    )
    print(f'  → Kept majority label; {len(df):,} rows remain')

In [ ]:
# ── 1.3  EDA plots ───────────────────────────────────────────────
df['text_len']   = df[TEXT_COL].apply(len)
df['word_count'] = df[TEXT_COL].apply(lambda x: len(x.split()))
langs = sorted(df[LANG_COL].unique())

fig = plt.figure(figsize=(20, 12))
fig.patch.set_facecolor('#0d1117')
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.5, wspace=0.38)

# Label bar
ax1 = fig.add_subplot(gs[0, 0])
lc  = df[LABEL_COL].value_counts().sort_index()
bars = ax1.bar(['Not Hate (0)', 'Hate (1)'],
               lc.values, color=[LABEL_CLR[i] for i in lc.index], edgecolor='none')
for b, v in zip(bars, lc.values):
    ax1.text(b.get_x()+b.get_width()/2, v+lc.max()*0.01,
             f'{v:,}', ha='center', va='bottom', fontsize=10)
ax1.axhline(lc.mean(), color='#f0f6fc', ls='--', lw=1.3,
            label=f'Mean={lc.mean():,.0f}')
ax1.set_title('Label Distribution', fontsize=13, pad=10)
ax1.set_ylabel('Count'); ax1.legend(fontsize=8); ax1.grid(axis='y', alpha=0.4)

# Language bar
ax2 = fig.add_subplot(gs[0, 1])
langc = df[LANG_COL].value_counts()
bars2 = ax2.bar(langc.index, langc.values,
                color=[LANG_CLR.get(l, PALETTE[0]) for l in langc.index],
                edgecolor='none')
for b, v in zip(bars2, langc.values):
    ax2.text(b.get_x()+b.get_width()/2, v+langc.max()*0.01,
             f'{v:,}', ha='center', va='bottom', fontsize=10)
ax2.set_title('Language Distribution', fontsize=13, pad=10)
ax2.set_ylabel('Count'); ax2.grid(axis='y', alpha=0.4)

# Language × Label
ax3 = fig.add_subplot(gs[0, 2])
cross = pd.crosstab(df[LANG_COL], df[LABEL_COL])
x, w = np.arange(len(cross.index)), 0.35
ax3.bar(x - w/2, cross.get(0, pd.Series([0]*len(cross))), w,
        color=LABEL_CLR[0], alpha=0.85, label='Not Hate')
ax3.bar(x + w/2, cross.get(1, pd.Series([0]*len(cross))), w,
        color=LABEL_CLR[1], alpha=0.85, label='Hate')
ax3.set_xticks(x); ax3.set_xticklabels(cross.index)
ax3.set_title('Label per Language', fontsize=13, pad=10)
ax3.set_ylabel('Count'); ax3.legend(fontsize=9); ax3.grid(axis='y', alpha=0.4)

# Text length violin
ax4 = fig.add_subplot(gs[1, 0])
vp_data = [df[df[LANG_COL] == l]['text_len'].values for l in langs]
vp = ax4.violinplot(vp_data, showmedians=True, showextrema=True)
for i, body in enumerate(vp['bodies']):
    body.set_facecolor(LANG_CLR.get(langs[i], PALETTE[i % len(PALETTE)]))
    body.set_alpha(0.7)
vp['cmedians'].set_color('#ffffff'); vp['cmedians'].set_linewidth(2)
ax4.set_xticks(range(1, len(langs)+1)); ax4.set_xticklabels(langs)
ax4.set_title('Text Length per Language (chars)', fontsize=12, pad=10)
ax4.set_ylabel('Characters'); ax4.grid(axis='y', alpha=0.3)

# Word count by label
ax5 = fig.add_subplot(gs[1, 1])
for lbl in [0, 1]:
    ax5.hist(df[df[LABEL_COL] == lbl]['word_count'], bins=40,
             alpha=0.6, color=LABEL_CLR[lbl], density=True, edgecolor='none',
             label='Not Hate' if lbl == 0 else 'Hate')
ax5.set_title('Word Count by Label (density)', fontsize=12, pad=10)
ax5.set_xlabel('Words'); ax5.set_ylabel('Density')
ax5.legend(fontsize=9); ax5.grid(alpha=0.3)

# Avg text length heatmap
ax6 = fig.add_subplot(gs[1, 2])
heat = df.groupby([LANG_COL, LABEL_COL])['text_len'].mean().unstack()
heat.columns = ['Not Hate', 'Hate']
sns.heatmap(heat, annot=True, fmt='.0f', cmap='Blues',
            linewidths=0.5, linecolor='#0d1117', ax=ax6,
            cbar_kws={'label': 'Avg chars'})
ax6.set_title('Avg Text Length — Lang × Label', fontsize=12, pad=10)
ax6.set_xlabel(''); ax6.set_ylabel('')

fig.suptitle('🌐  Trilingual Hate Speech Dataset — EDA',
             fontsize=16, y=1.01, color='#f0f6fc', fontweight='bold')
plt.savefig('fig1_eda.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Figure 1 saved')

In [ ]:
# ── Figure 2 ─ Word Clouds (lang × label) ────────────────────────
STOPWORDS = {
    # English
    'the','a','an','is','it','in','on','at','to','of','and','or','for','this',
    'that','was','with','are','be','have','as','from','by','not','but','so',
    'if','do','he','she','they','we','you','i','me','my','http','https','www',
    'co','rt','amp','via','user',
    # French
    'le','la','les','de','du','des','un','une','en','et','est','que','qui',
    'pas','je','tu','il','elle','nous','vous','ils','elles','ce','se','sa','son',
    'sur','au','aux','par','pour','dans','avec','ne','plus','tout',
    # Arabic
    'في','من','على','إلى','هذا','هذه','التي','الذي','كان','كانت','مع','عن',
    'أن','لا','ما','هو','هي','هم','نحن','انت','انا','عليه','عليها'
}
cmaps_map = {
    ('en', 0): 'Blues',   ('en', 1): 'Reds',
    ('fr', 0): 'Purples', ('fr', 1): 'Oranges',
    ('ar', 0): 'YlGn',    ('ar', 1): 'YlOrRd',
}
label_names = {0: 'Not Hate', 1: 'Hate'}

fig, axes = plt.subplots(len(langs), 2, figsize=(16, 5*len(langs)))
fig.patch.set_facecolor('#0d1117')
if len(langs) == 1:
    axes = [axes]

for row, lang in enumerate(langs):
    for col, lbl in enumerate([0, 1]):
        ax = axes[row][col]
        subset = df[(df[LANG_COL] == lang) & (df[LABEL_COL] == lbl)][TEXT_COL]
        if len(subset) == 0:
            ax.axis('off'); continue
        corpus = ' '.join(subset.tolist())
        corpus = re.sub(r'http\S+|www\S+|@\w+', ' ', corpus)
        corpus = re.sub(r'\s+', ' ', corpus).strip()
        words  = [w.lower() for w in corpus.split()
                  if w.lower() not in STOPWORDS and len(w) > 1]
        if not words:
            ax.axis('off'); continue
        wc = WordCloud(
            width=700, height=350,
            background_color='#0d1117',
            colormap=cmaps_map.get((lang, lbl), 'viridis'),
            max_words=100, prefer_horizontal=0.75,
            regexp=r'[\w\u0600-\u06ff]+',
        ).generate(' '.join(words))
        ax.imshow(wc, interpolation='bilinear')
        ax.axis('off')
        ax.set_title(f'{lang.upper()} — {label_names[lbl]}',
                     fontsize=13, color='#f0f6fc', pad=8)

fig.suptitle('☁️  Word Clouds — Language × Label',
             fontsize=16, color='#f0f6fc', fontweight='bold')
plt.tight_layout()
plt.savefig('fig2_wordclouds.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Figure 2 saved')

---
## 🔍 2. Leakage Audit

This section checks for features that could make the task artificially easy — the most common cause of suspiciously perfect scores.

In [ ]:
# ── 2.1  Check for non-text columns that might leak the label ────
print('Column types:')
print(df.dtypes)
print()

# Drop any column that is NOT text, label, or lang
keep_cols = [TEXT_COL, LABEL_COL, LANG_COL, 'text_len', 'word_count']
extra_cols = [c for c in df.columns if c not in keep_cols]
if extra_cols:
    print(f'⚠️  Extra columns found and DROPPED: {extra_cols}')
    df = df.drop(columns=extra_cols)
else:
    print('✅ No suspicious extra columns')

# ── 2.2  Check label-text correlation via length ─────────────────
print()
print('Text length stats per label:')
print(df.groupby(LABEL_COL)['text_len'].describe().round(1))
print()
# If hate texts are ALWAYS much shorter/longer → length alone predicts label → leak
len_corr = df['text_len'].corr(df[LABEL_COL])
print(f'Correlation(text_len, label) = {len_corr:.4f}')
if abs(len_corr) > 0.5:
    print('  ⚠️  HIGH CORRELATION — text length is a strong label predictor!')
    print('      This may indicate upsampling artifacts. Proceed with caution.')
else:
    print('  ✅ Low correlation — text length is not a leaky feature')

In [ ]:
# ── 2.3  Check for near-duplicate texts across label classes ─────
# If the same text appears with label 0 in train and label 1 in test → leakage
conflict_texts = (
    df.groupby(TEXT_COL)[LABEL_COL]
      .nunique()
      .loc[lambda s: s > 1]
)
print(f'Texts with conflicting labels : {len(conflict_texts):,}')
if len(conflict_texts) > 0:
    print('  Sample conflicting texts:')
    sample = conflict_texts.head(3).index.tolist()
    for t in sample:
        print(f'    "{t[:80]}"')
    # Remove ALL rows with conflicting texts to prevent leakage
    df = df[~df[TEXT_COL].isin(conflict_texts.index)].reset_index(drop=True)
    print(f'  → All conflicting texts removed. {len(df):,} rows remain.')
else:
    print('  ✅ No conflicting labels found')

print(f'\nFinal dataset size : {len(df):,}')
print('Final label distribution:')
print(df[LABEL_COL].value_counts().sort_index())

---
## 🔧 3. Preprocessing

In [ ]:
# ── 3.1  Minimal multilingual-safe text cleaning ─────────────────
# KEY PRINCIPLE: We clean noise (URLs, extra spaces) but we do NOT remove
# words or punctuation that could carry hate-speech signal.
# mBERT's WordPiece tokenizer handles Arabic script and French accents natively.

def clean_text(text: str) -> str:
    text = str(text)
    text = re.sub(r'http\S+|www\S+', ' ', text)   # Remove URLs (no semantic content)
    text = re.sub(r'@\w+', '@user', text)           # Anonymise mentions
    text = re.sub(r'#(\w+)', r'\1', text)           # Hashtag → plain word
    text = re.sub(r'(.)\1{3,}', r'\1\1', text)      # Normalise elongation (haaaate→haate)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['clean_text'] = df[TEXT_COL].apply(clean_text)

# Sanity check — show one sample per language
print('Cleaned text samples:')
for lang in langs:
    row = df[df[LANG_COL] == lang].iloc[0]
    print(f'  [{lang}] {row["clean_text"][:90]}')

In [ ]:
# ── 3.2  Stratified split — BEFORE any further processing ────────
# Stratify by label to maintain class balance in every split.
# 70 / 15 / 15

train_df, temp_df = train_test_split(
    df, test_size=0.30, random_state=SEED, stratify=df[LABEL_COL]
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, random_state=SEED, stratify=temp_df[LABEL_COL]
)
for split in [train_df, val_df, test_df]:
    split.reset_index(drop=True, inplace=True)

print(f'Train : {len(train_df):>7,}  ({len(train_df)/len(df)*100:.1f}%)')
print(f'Val   : {len(val_df):>7,}  ({len(val_df)/len(df)*100:.1f}%)')
print(f'Test  : {len(test_df):>7,}  ({len(test_df)/len(df)*100:.1f}%)')

print('\nLabel balance per split:')
for name, split in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
    vc = split[LABEL_COL].value_counts(normalize=True).sort_index()
    print(f'  {name}: not-hate={vc.get(0,0)*100:.1f}%  hate={vc.get(1,0)*100:.1f}%')

print('\nLanguage coverage per split:')
for name, split in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
    print(f'  {name}: {dict(split[LANG_COL].value_counts())}')

In [ ]:
# ── 3.3  Tokenizer ────────────────────────────────────────────────
MODEL_NAME = 'bert-base-multilingual-cased'
MAX_LEN    = 128

tokenizer = BertTokenizerFast.from_pretrained(MODEL_NAME)
print(f'✅ Tokenizer: {MODEL_NAME}  |  vocab={tokenizer.vocab_size:,}')

print('\nToken length coverage at MAX_LEN=128:')
for lang in langs:
    sample = df[df[LANG_COL] == lang]['clean_text'].sample(
        min(500, (df[LANG_COL] == lang).sum()), random_state=SEED
    )
    lens = sample.apply(
        lambda x: len(tokenizer.encode(x, add_special_tokens=True))
    )
    cov = (lens <= MAX_LEN).mean()
    print(f'  {lang.upper()}: mean={lens.mean():.0f}  '
          f'p95={lens.quantile(0.95):.0f}  coverage={cov*100:.1f}%')

In [ ]:
# ── 3.4  PyTorch Dataset ─────────────────────────────────────────
class HateSpeechDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts     = list(texts)
        self.labels    = list(labels)
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            str(self.texts[idx]),
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt',
        )
        return {
            'input_ids'      : enc['input_ids'].squeeze(0),
            'attention_mask' : enc['attention_mask'].squeeze(0),
            'token_type_ids' : enc['token_type_ids'].squeeze(0),
            'labels'         : torch.tensor(int(self.labels[idx]), dtype=torch.long),
        }

# Effective batch = 32 × N_GPUS
BATCH_PER_GPU = 32
BATCH_SIZE    = BATCH_PER_GPU * max(1, N_GPUS)

train_ds = HateSpeechDataset(train_df['clean_text'], train_df[LABEL_COL], tokenizer, MAX_LEN)
val_ds   = HateSpeechDataset(val_df['clean_text'],   val_df[LABEL_COL],   tokenizer, MAX_LEN)
test_ds  = HateSpeechDataset(test_df['clean_text'],  test_df[LABEL_COL],  tokenizer, MAX_LEN)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=4, pin_memory=True)

print(f'✅ DataLoaders ready')
print(f'   Effective batch size : {BATCH_SIZE}  ({BATCH_PER_GPU} × {N_GPUS} GPU)')
print(f'   Train / Val / Test   : {len(train_loader)} / {len(val_loader)} / {len(test_loader)} batches')

---
## 🤖 4. Fine-Tuning

### Hyperparameters chosen to prevent overfitting

| Param | Value | Why |
|---|---|---|
| LR (encoder) | **1e-5** | Half of standard — slower, less memorisation |
| LR (head) | **1e-4** | 10× encoder; head is randomly initialised |
| Classifier dropout | **0.3** | Up from BERT's default 0.1 — stronger regularisation |
| Weight decay | **0.05** | L2 penalty on all non-bias weights |
| Warmup | **15%** | Longer ramp prevents aggressive early updates |
| Max epochs | **6** | Upper bound only — early stopping will kick in sooner |
| Early stopping patience | **2** | Stop if val F1 doesn't improve for 2 consecutive epochs |

In [ ]:
# ── 4.1  Build model with higher dropout ─────────────────────────
# id2label / label2id must use native Python int/str (not numpy) to avoid JSON error
id2label  = {0: 'not_hate', 1: 'hate'}
label2id  = {'not_hate': 0, 'hate': 1}
NUM_LABELS = 2

# Load config first so we can override classifier_dropout
config = BertConfig.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=id2label,
    label2id=label2id,
    hidden_dropout_prob=0.2,        # slightly higher than default 0.1
    attention_probs_dropout_prob=0.2,
    classifier_dropout=0.3,         # extra dropout before the classification head
)

model = BertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    config=config,
    ignore_mismatched_sizes=True,
)

if N_GPUS > 1:
    model = nn.DataParallel(model)
    print(f'✅ DataParallel across {N_GPUS} GPUs')
model = model.to(DEVICE)

total_p = sum(p.numel() for p in model.parameters())
print(f'   Total params : {total_p/1e6:.1f}M')

In [ ]:
# ── 4.2  Optimizer + Scheduler + Scaler ──────────────────────────
EPOCHS       = 6       # Upper bound — early stopping will usually halt before this
LR_ENCODER   = 1e-5   # Conservative LR for pre-trained encoder
LR_HEAD      = 1e-4   # Higher LR for randomly-init classifier head
WARMUP_RATIO = 0.15
WEIGHT_DECAY = 0.05
PATIENCE     = 2       # Early stopping patience (epochs)

total_steps  = len(train_loader) * EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)

core_model = model.module if isinstance(model, nn.DataParallel) else model

# No-decay list: bias + LayerNorm weights should not be penalised
no_decay = ['bias', 'LayerNorm.weight']
optimizer_groups = [
    # BERT encoder — with decay, lower LR
    {'params': [p for n, p in core_model.bert.named_parameters()
                if not any(nd in n for nd in no_decay)],
     'lr': LR_ENCODER, 'weight_decay': WEIGHT_DECAY},
    {'params': [p for n, p in core_model.bert.named_parameters()
                if any(nd in n for nd in no_decay)],
     'lr': LR_ENCODER, 'weight_decay': 0.0},
    # Classifier head — with decay, higher LR
    {'params': [p for n, p in core_model.classifier.named_parameters()
                if not any(nd in n for nd in no_decay)],
     'lr': LR_HEAD, 'weight_decay': WEIGHT_DECAY},
    {'params': [p for n, p in core_model.classifier.named_parameters()
                if any(nd in n for nd in no_decay)],
     'lr': LR_HEAD, 'weight_decay': 0.0},
]

optimizer = AdamW(optimizer_groups, eps=1e-8)
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps
)
scaler = GradScaler()

print('✅ Optimizer ready')
print(f'   LR encoder   : {LR_ENCODER}   LR head : {LR_HEAD}')
print(f'   Weight decay : {WEIGHT_DECAY}  Dropout : 0.3')
print(f'   Warmup steps : {warmup_steps}   Total   : {total_steps}')
print(f'   Early stop   : patience={PATIENCE} epochs on val F1')

In [ ]:
# ── 4.3  Train / Eval functions ───────────────────────────────────
def run_train_epoch(model, loader, optimizer, scheduler, scaler, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for batch in loader:
        ids   = batch['input_ids'].to(device)
        mask  = batch['attention_mask'].to(device)
        ttype = batch['token_type_ids'].to(device)
        labs  = batch['labels'].to(device)

        optimizer.zero_grad(set_to_none=True)
        with autocast():
            out  = model(input_ids=ids, attention_mask=mask,
                         token_type_ids=ttype, labels=labs)
            loss = out.loss.mean() if out.loss.dim() > 0 else out.loss

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        total_loss += loss.item()
        preds = out.logits.argmax(dim=-1)
        correct += (preds == labs).sum().item()
        total   += labs.size(0)
    return total_loss / len(loader), correct / total


@torch.no_grad()
def run_eval_epoch(model, loader, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels, all_probs = [], [], []
    for batch in loader:
        ids   = batch['input_ids'].to(device)
        mask  = batch['attention_mask'].to(device)
        ttype = batch['token_type_ids'].to(device)
        labs  = batch['labels'].to(device)
        with autocast():
            out  = model(input_ids=ids, attention_mask=mask,
                         token_type_ids=ttype, labels=labs)
            loss = out.loss.mean() if out.loss.dim() > 0 else out.loss
        total_loss += loss.item()
        probs = torch.softmax(out.logits, dim=-1)
        preds = probs.argmax(dim=-1)
        correct += (preds == labs).sum().item()
        total   += labs.size(0)
        all_preds.extend(preds.cpu().numpy().tolist())
        all_labels.extend(labs.cpu().numpy().tolist())
        all_probs.extend(probs.cpu().numpy().tolist())
    return (total_loss / len(loader), correct / total,
            all_preds, all_labels, all_probs)

print('✅ Training functions defined')

In [ ]:
# ── 4.4  Training loop with early stopping ───────────────────────
history = {
    'train_loss': [], 'val_loss': [],
    'train_acc':  [], 'val_acc':  [],
    'val_f1':     [], 'val_auc':  [],
}
best_val_f1   = 0.0
best_epoch    = 0
patience_ctr  = 0
CKPT_PATH     = 'best_mbert_hatespeech.pt'

print('Starting training...\n')
hdr = (f"{'Ep':>3} | {'TrLoss':>8} | {'TrAcc':>7} | "
       f"{'VlLoss':>8} | {'VlAcc':>7} | {'F1':>7} | {'AUC':>7} | {'Note'}")
print(hdr)
print('-' * len(hdr))

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = run_train_epoch(
        model, train_loader, optimizer, scheduler, scaler, DEVICE
    )
    vl_loss, vl_acc, vl_preds, vl_labs, vl_probs = run_eval_epoch(
        model, val_loader, DEVICE
    )
    vl_f1  = f1_score(vl_labs, vl_preds, average='binary', zero_division=0)
    vl_auc = roc_auc_score(vl_labs, [p[1] for p in vl_probs])

    history['train_loss'].append(tr_loss)
    history['val_loss'].append(vl_loss)
    history['train_acc'].append(tr_acc)
    history['val_acc'].append(vl_acc)
    history['val_f1'].append(vl_f1)
    history['val_auc'].append(vl_auc)

    # ── Overfitting warning ───────────────────────────────────────
    gap = tr_acc - vl_acc
    if tr_acc > 0.99 and gap > 0.05:
        note = f'⚠️  overfit gap={gap:.3f}'
    elif vl_f1 > best_val_f1:
        note = '✅ best'
    else:
        note = f'patience {patience_ctr+1}/{PATIENCE}'

    print(f"{epoch:>3} | {tr_loss:>8.4f} | {tr_acc*100:>6.2f}% | "
          f"{vl_loss:>8.4f} | {vl_acc*100:>6.2f}% | "
          f"{vl_f1:.4f} | {vl_auc:.4f} | {note}")

    # ── Save best checkpoint ──────────────────────────────────────
    if vl_f1 > best_val_f1:
        best_val_f1  = vl_f1
        best_epoch   = epoch
        patience_ctr = 0
        core = model.module if isinstance(model, nn.DataParallel) else model
        torch.save(core.state_dict(), CKPT_PATH)
    else:
        patience_ctr += 1
        if patience_ctr >= PATIENCE:
            print(f'\n⏹️  Early stopping triggered at epoch {epoch} '
                  f'(no improvement for {PATIENCE} epochs)')
            break

print(f'\n🏆 Best checkpoint → Epoch {best_epoch}  |  Val F1 = {best_val_f1:.4f}')

In [ ]:
# ── Figure 3 ─ Learning curves ────────────────────────────────────
n_ep = len(history['train_loss'])
ep_r = range(1, n_ep + 1)

fig, axes = plt.subplots(1, 4, figsize=(22, 5))
fig.patch.set_facecolor('#0d1117')

specs = [
    ('Loss',         history['train_loss'],           history['val_loss'],   False),
    ('Accuracy (%)', [x*100 for x in history['train_acc']],
                     [x*100 for x in history['val_acc']],                    False),
    ('Val F1',       None,                             history['val_f1'],    True),
    ('Val AUC',      None,                             history['val_auc'],   True),
]
for ax, (title, tr_v, vl_v, val_only) in zip(axes, specs):
    if not val_only:
        ax.plot(ep_r, tr_v, 'o-', color=C['blue'],  lw=2, label='Train')
    ax.plot(ep_r, vl_v, 's-', color=C['green'], lw=2, label='Val')
    best_x = int(np.argmax(vl_v)) + 1
    ax.axvline(best_x, ls='--', color=C['orange'], lw=1.5, label=f'Best E{best_x}')
    ax.set_title(title, fontsize=13, pad=8)
    ax.set_xlabel('Epoch')
    ax.set_xticks(list(ep_r))
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

# Shade gap between train and val accuracy to visualise overfitting
tr_pct = [x*100 for x in history['train_acc']]
vl_pct = [x*100 for x in history['val_acc']]
axes[1].fill_between(ep_r, tr_pct, vl_pct,
                     where=[t > v for t, v in zip(tr_pct, vl_pct)],
                     alpha=0.15, color=C['red'], label='Overfit gap')
axes[1].legend(fontsize=7)

fig.suptitle('📉  Learning Curves (with overfitting gap)',
             fontsize=16, color='#f0f6fc', fontweight='bold')
plt.tight_layout()
plt.savefig('fig3_learning_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Figure 3 saved')

---
## 📈 5. Evaluation & Analysis

In [ ]:
# ── 5.1  Load best checkpoint → run on test set ──────────────────
core_model = model.module if isinstance(model, nn.DataParallel) else model
core_model.load_state_dict(torch.load(CKPT_PATH, map_location=DEVICE))
model.eval()

_, test_acc, test_preds, test_labels, test_probs = run_eval_epoch(
    model, test_loader, DEVICE
)
test_probs_arr = np.array(test_probs)
test_probs_pos = test_probs_arr[:, 1]

test_f1  = f1_score(test_labels, test_preds, average='binary', zero_division=0)
test_auc = roc_auc_score(test_labels, test_probs_pos)
test_mcc = matthews_corrcoef(test_labels, test_preds)
test_ap  = average_precision_score(test_labels, test_probs_pos)

print('=' * 58)
print(' TEST SET RESULTS — mBERT Trilingual')
print('=' * 58)
print(f'  Accuracy       : {test_acc*100:.2f}%')
print(f'  F1 (binary)    : {test_f1:.4f}')
print(f'  AUC-ROC        : {test_auc:.4f}')
print(f'  Avg Precision  : {test_ap:.4f}')
print(f'  MCC            : {test_mcc:.4f}')
print()
# Flag if results still look suspiciously perfect
if test_acc > 0.99:
    print('⚠️  Accuracy >99% — still suspicious. Re-examine dataset for leakage.')
elif test_acc > 0.95:
    print('✅ Strong results — verify with per-language breakdown below.')
else:
    print('✅ Results look realistic.')
print()
print('Detailed Classification Report:')
print(classification_report(
    test_labels, test_preds,
    target_names=['not_hate', 'hate'], digits=4
))

In [ ]:
# ── Figure 4 ─ Confusion matrices ────────────────────────────────
cm      = confusion_matrix(test_labels, test_preds)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.patch.set_facecolor('#0d1117')

for ax, data, title, fmt in [
    (axes[0], cm,      'Counts',     'd'),
    (axes[1], cm_norm, 'Normalised', '.2f'),
]:
    im = ax.imshow(data, cmap='magma', aspect='auto', vmin=0, vmax=data.max())
    ax.set_xticks([0, 1]); ax.set_xticklabels(['Not Hate', 'Hate'])
    ax.set_yticks([0, 1]); ax.set_yticklabels(['Not Hate', 'Hate'])
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
    ax.set_title(f'Confusion Matrix — {title}', fontsize=13, pad=8)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    thresh = data.max() / 2.0
    for i in range(2):
        for j in range(2):
            ax.text(j, i, format(data[i, j], fmt),
                    ha='center', va='center', fontsize=14,
                    color='white' if data[i, j] < thresh else 'black')

fig.suptitle('🎯  Confusion Matrices', fontsize=16, color='#f0f6fc', fontweight='bold')
plt.tight_layout()
plt.savefig('fig4_confusion.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Figure 4 saved')

In [ ]:
# ── Figure 5 ─ ROC + Precision-Recall curves ─────────────────────
fpr, tpr, roc_thresholds = roc_curve(test_labels, test_probs_pos)
prec, rec, pr_thresholds = precision_recall_curve(test_labels, test_probs_pos)

# Optimal threshold via Youden's J
j_scores = tpr - fpr
opt_idx  = int(np.argmax(j_scores))
opt_thr  = float(roc_thresholds[opt_idx])

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.patch.set_facecolor('#0d1117')
for ax in axes: ax.set_facecolor('#0d1117')

# ROC
axes[0].plot(fpr, tpr, color=C['blue'], lw=2.5,
             label=f'mBERT  (AUC={test_auc:.3f})')
axes[0].scatter([fpr[opt_idx]], [tpr[opt_idx]],
                color=C['orange'], s=100, zorder=5,
                label=f'Optimal thr={opt_thr:.3f}')
axes[0].fill_between(fpr, tpr, alpha=0.08, color=C['blue'])
axes[0].plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5)
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve', fontsize=13, pad=8)
axes[0].legend(fontsize=10); axes[0].grid(alpha=0.25)

# Precision-Recall
axes[1].plot(rec, prec, color=C['green'], lw=2.5,
             label=f'mBERT  (AP={test_ap:.3f})')
baseline = float(np.mean(test_labels))
axes[1].axhline(baseline, color='k', ls='--', lw=1, alpha=0.5,
                label=f'Baseline  (AP={baseline:.3f})')
axes[1].fill_between(rec, prec, alpha=0.08, color=C['green'])
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve', fontsize=13, pad=8)
axes[1].legend(fontsize=10); axes[1].grid(alpha=0.25)

fig.suptitle('📐  ROC + Precision-Recall Curves',
             fontsize=16, color='#f0f6fc', fontweight='bold')
plt.tight_layout()
plt.savefig('fig5_roc_pr.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ Figure 5 saved  |  Optimal threshold = {opt_thr:.3f}')

In [ ]:
# ── Figure 6 ─ Per-language performance breakdown ─────────────────
test_df = test_df.copy()
test_df['pred']      = test_preds
test_df['prob_hate'] = test_probs_pos
test_df['correct']   = (test_df['pred'] == test_df[LABEL_COL]).astype(int)

lang_metrics = {}
for lang in langs:
    sub = test_df[test_df[LANG_COL] == lang]
    if len(sub) == 0:
        continue
    labs_  = sub[LABEL_COL].tolist()
    preds_ = sub['pred'].tolist()
    probs_ = sub['prob_hate'].tolist()
    lang_metrics[lang] = {
        'n'  : len(sub),
        'acc': accuracy_score(labs_, preds_),
        'f1' : f1_score(labs_, preds_, average='binary', zero_division=0),
        'auc': roc_auc_score(labs_, probs_) if len(set(labs_)) > 1 else float('nan'),
        'mcc': matthews_corrcoef(labs_, preds_),
    }

metrics_df = pd.DataFrame(lang_metrics).T
print('Per-language test metrics:')
print(metrics_df.round(4))

metric_cols = ['acc', 'f1', 'auc', 'mcc']
metric_lbls = ['Accuracy', 'F1', 'AUC-ROC', 'MCC']
x = np.arange(len(langs))
w = 0.18

fig, ax = plt.subplots(figsize=(12, 6))
fig.patch.set_facecolor('#0d1117'); ax.set_facecolor('#0d1117')

for i, (col, lbl) in enumerate(zip(metric_cols, metric_lbls)):
    vals = [lang_metrics.get(lang, {}).get(col, 0) for lang in langs]
    offset = (i - len(metric_cols)/2 + 0.5) * w
    bars = ax.bar(x + offset, vals, w, label=lbl,
                  color=PALETTE[i], alpha=0.85, edgecolor='none')
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, v+0.005,
                f'{v:.2f}', ha='center', va='bottom', fontsize=7, rotation=45)

ax.set_xticks(x); ax.set_xticklabels([l.upper() for l in langs])
ax.set_ylim(0, 1.18); ax.set_ylabel('Score')
ax.set_title('Per-Language Performance', fontsize=14, pad=10)
ax.legend(fontsize=9); ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('fig6_per_lang.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Figure 6 saved')

In [ ]:
# ── Figure 7 ─ Confidence analysis ───────────────────────────────
max_conf     = test_probs_arr.max(axis=1)
correct_mask = np.array(test_preds) == np.array(test_labels)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.patch.set_facecolor('#0d1117')

# Correct vs incorrect confidence
axes[0].hist(max_conf[correct_mask],  bins=40, alpha=0.7, color=C['green'],
             label=f'Correct ({correct_mask.sum():,})',
             edgecolor='none', density=True)
axes[0].hist(max_conf[~correct_mask], bins=40, alpha=0.7, color=C['red'],
             label=f'Incorrect ({(~correct_mask).sum():,})',
             edgecolor='none', density=True)
axes[0].set_title('Confidence — Correct vs Incorrect', fontsize=12)
axes[0].set_xlabel('Max Softmax Probability')
axes[0].set_ylabel('Density'); axes[0].legend(); axes[0].grid(alpha=0.3)

# Confidence per language
lang_conf = [max_conf[test_df[LANG_COL].values == lang] for lang in langs]
bp = axes[1].boxplot(lang_conf, patch_artist=True, notch=True,
                     medianprops={'color': '#f0f6fc', 'linewidth': 2})
for patch, lang in zip(bp['boxes'], langs):
    patch.set_facecolor(LANG_CLR.get(lang, PALETTE[0])); patch.set_alpha(0.7)
axes[1].set_xticks(range(1, len(langs)+1))
axes[1].set_xticklabels([l.upper() for l in langs])
axes[1].set_title('Confidence per Language', fontsize=12)
axes[1].set_ylabel('Max Softmax Probability'); axes[1].grid(axis='y', alpha=0.3)

# Reliability diagram
bins_ = np.linspace(0, 1, 11)
bucket_acc_, bucket_mid_ = [], []
for lo, hi in zip(bins_[:-1], bins_[1:]):
    m = (max_conf >= lo) & (max_conf < hi)
    if m.sum() > 0:
        bucket_acc_.append(float(correct_mask[m].mean()))
        bucket_mid_.append(float((lo + hi) / 2))
axes[2].bar(bucket_mid_, bucket_acc_, width=0.08,
            color=C['blue'], alpha=0.8, edgecolor='none')
axes[2].plot([0, 1], [0, 1], 'k--', lw=1.5, label='Perfect calibration')
axes[2].set_title('Reliability Diagram', fontsize=12)
axes[2].set_xlabel('Confidence Bucket')
axes[2].set_ylabel('Accuracy'); axes[2].legend(fontsize=9); axes[2].grid(alpha=0.3)

fig.suptitle('🎰  Prediction Confidence Analysis',
             fontsize=16, color='#f0f6fc', fontweight='bold')
plt.tight_layout()
plt.savefig('fig7_confidence.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Figure 7 saved')

In [ ]:
# ── Figure 8 ─ Error analysis per language ────────────────────────
err_df = test_df[test_df['correct'] == 0].copy()

err_cross = pd.crosstab(err_df[LANG_COL], err_df[LABEL_COL])
for c in [0, 1]:
    if c not in err_cross.columns:
        err_cross[c] = 0
err_cross = err_cross[[0, 1]]
err_cross.columns = ['FN (Hate→Not Hate)', 'FP (Not Hate→Hate)']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#0d1117')

err_cross.plot(kind='bar', ax=axes[0],
               color=[C['green'], C['red']],
               edgecolor='none', alpha=0.85, rot=0)
axes[0].set_title('Error Types per Language', fontsize=13, pad=10)
axes[0].set_xlabel('Language'); axes[0].set_ylabel('Count')
axes[0].legend(fontsize=9); axes[0].grid(axis='y', alpha=0.3)

err_rate = test_df.groupby(LANG_COL)['correct'].apply(lambda x: 1 - x.mean())
axes[1].bar(err_rate.index, err_rate.values * 100,
            color=[LANG_CLR.get(l, PALETTE[0]) for l in err_rate.index],
            edgecolor='none', alpha=0.85)
for i, (lang, val) in enumerate(err_rate.items()):
    axes[1].text(i, val*100 + 0.2, f'{val*100:.2f}%',
                 ha='center', va='bottom', fontsize=10)
axes[1].set_title('Error Rate per Language (%)', fontsize=13, pad=10)
axes[1].set_xlabel('Language'); axes[1].set_ylabel('Error Rate (%)')
axes[1].grid(axis='y', alpha=0.3)

fig.suptitle('🔍  Error Analysis — Per Language',
             fontsize=16, color='#f0f6fc', fontweight='bold')
plt.tight_layout()
plt.savefig('fig8_errors.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Total errors : {len(err_df):,} / {len(test_df):,} '
      f'({len(err_df)/len(test_df)*100:.1f}%)')
print('\nError breakdown:')
print(err_cross)
print('✅ Figure 8 saved')

In [ ]:
# ── 5.6  Most confident wrong predictions ────────────────────────
err_sorted = test_df[test_df['correct'] == 0].copy()
err_sorted['conf'] = max_conf[test_df['correct'].values == 0]
err_sorted = err_sorted.sort_values('conf', ascending=False)

print('Top-10 most confident wrong predictions:\n')
for _, row in err_sorted.head(10).iterrows():
    true_name = id2label[int(row[LABEL_COL])]
    pred_name = id2label[int(row['pred'])]
    print(f"  [{row[LANG_COL].upper()}] True={true_name:<10} "
          f"Pred={pred_name:<10} Conf={row['conf']:.3f}")
    print(f"         {str(row['clean_text'])[:90]}")
    print()

In [ ]:
# ── 5.7  Save model + metadata ───────────────────────────────────
SAVE_DIR = 'mbert_trilingual_hatespeech'
core_model = model.module if isinstance(model, nn.DataParallel) else model
core_model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

meta = {
    'model'             : MODEL_NAME,
    'task'              : 'binary hate speech detection',
    'languages'         : langs,
    'label_map'         : id2label,
    'max_len'           : MAX_LEN,
    'test_accuracy'     : round(float(test_acc),  4),
    'test_f1'           : round(float(test_f1),   4),
    'test_auc'          : round(float(test_auc),  4),
    'test_avg_precision': round(float(test_ap),   4),
    'test_mcc'          : round(float(test_mcc),  4),
    'optimal_threshold' : round(float(opt_thr),   4),
    'best_epoch'        : int(best_epoch),
    'epochs_trained'    : int(n_ep),
}
with open(f'{SAVE_DIR}/training_meta.json', 'w') as f:
    json.dump(meta, f, indent=2)

print('\n' + '=' * 58)
print(' FINAL SUMMARY')
print('=' * 58)
for k, v in meta.items():
    print(f'  {k:<24}: {v}')
print('=' * 58)
print(f'\n✅ Model saved → ./{SAVE_DIR}/')
print('📊 Figures : fig1_eda … fig8_errors.png')

---
## 🔮 6. Inference Demo

In [ ]:
def predict(text: str, threshold: float = 0.5) -> dict:
    """
    Classify a single text as hate / not-hate.
    Works for EN / FR / AR — mBERT handles language internally.
    threshold: float between 0-1, default 0.5 (use opt_thr for Youden-optimal)
    """
    core = model.module if isinstance(model, nn.DataParallel) else model
    core.eval()
    enc = tokenizer(
        clean_text(text),
        max_length=MAX_LEN, padding='max_length',
        truncation=True, return_tensors='pt',
    )
    with torch.no_grad():
        logits = core(
            input_ids=enc['input_ids'].to(DEVICE),
            attention_mask=enc['attention_mask'].to(DEVICE),
            token_type_ids=enc['token_type_ids'].to(DEVICE),
        ).logits
    probs   = torch.softmax(logits, dim=-1).cpu().numpy()[0]
    pred_id = int(probs[1] >= threshold)
    return {
        'label'      : id2label[pred_id],
        'confidence' : round(float(probs[pred_id]) * 100, 2),
        'p_not_hate' : round(float(probs[0]) * 100, 2),
        'p_hate'     : round(float(probs[1]) * 100, 2),
    }

demos = [
    ('en', 'I think everyone deserves equal rights and respect.'),
    ('en', 'People like you are a disgrace and should be removed.'),
    ('en', 'This product is absolutely terrible, total waste of money.'),  # offensive but not hate
    ('fr', 'Nous devons respecter les droits de chacun.'),
    ('fr', 'Ces étrangers volent nos emplois et ruinent notre pays.'),
    ('ar', 'نحن نحب جميع الناس ونحترم الاختلاف.'),
    ('ar', 'هؤلاء الناس لا يستحقون العيش في بلدنا.'),
    ('ar', 'يا حمار هذا الكلام غلط'),  # insult but no group-based hate
]

print(f'\n🔮 Inference Demo  |  Threshold = {opt_thr:.3f} (Youden optimal)')
print('-' * 72)
for lang, text in demos:
    r    = predict(text, threshold=opt_thr)
    flag = '🚨 HATE    ' if r['label'] == 'hate' else '✅ NOT HATE'
    print(f'{flag} [{lang.upper()}]  conf={r["confidence"]}%  '
          f'hate={r["p_hate"]}%')
    print(f'  "{text[:80]}"')
    print()